# Chemeleon experiment results analysis 

(This is the main notebook for the final analysis after running all Chemeleon experiments on HPC)

In [9]:
import pandas as pd
import numpy as np
from scipy import stats


In [4]:
results_df = pd.read_csv("results/chemeleon/run2_results_hpc/all_runs2.csv")
per_target_results = pd.read_csv("results/chemeleon/run2_results_hpc//per_target2.csv")

results_df = results_df.fillna('NA')
per_target_results = per_target_results.fillna("NA")

results_df.head(2)

,task,exp,seed,freeze_encoder,encoder_lr,head_lr,readout,hidden_dim,num_layers,dropout,...,lr_factor,lr_patience,val_score,baseline_score,best_epoch,epochs_run,final_score,train_loss_first,train_loss_last,overrides
0,classification,r2/01_frozen_linear,42,True,0.00001,0.001,linear,NA,NA,NA,...,0.5,2,0.149956,0.0,20,20,0.149956,0.533532,0.428361,model=chemeleon task=classification readout=li...
1,classification,r2/01_frozen_linear,43,True,0.00001,0.001,linear,NA,NA,NA,...,0.5,2,0.147442,0.0,19,20,0.103199,0.534993,0.427529,model=chemeleon task=classification readout=li...


In [5]:
per_target_results.head(2)

,task,exp,seed,target,MCC,AUROC,AP,n,positive_fraction,MAE,R2,ST-RAE
0,classification,r2/01_frozen_linear,42,CYP2D6_is_TDI,0.10759,0.685011,0.46434,230,0.273913,NA,NA,NA
1,classification,r2/01_frozen_linear,42,CYP3A4_is_TDI,0.192322,0.76887,0.450967,530,0.203774,NA,NA,NA


## Overall experiments summary

In [8]:
results_df.groupby("task").agg(
    runs = ("val_score", "size"),
    n_config = ("exp", "nunique"),
    n_seeds = ('seed', 'nunique'),
    min_score = ('val_score', 'min'),
    max_score = ("val_score",'max'),
    mean_score = ('val_score',"mean"),
    baseline_mean = ('baseline_score', 'mean')
)

,runs,n_config,n_seeds,min_score,max_score,mean_score,baseline_mean
task,,,,,,,
classification,39,13,3,0.143572,0.314833,0.264174,0.000000
regression,39,13,3,0.703773,0.750938,0.727459,0.983028


our experiments consisted of 78 runs - 39 per task with 13 unique configurations (repeated for each of the 3 seeds)

For classification we see quite a big spread of validation scores with min being 0.14 and max being 0.31. For regression the difference is not that significant. 

For both tasks Chemeleon scores are better than the naive baseline predictor.

## How the results between configurations differ?

Since in the current setup the model is non deterministic we repeated each experiment (configuration) on 3 seeds. Now we will test if the differences between configurations are significant, or are they comparable to the noise (as changing seeds). 

Anova one way test - H0: the distributions dont differ significantly.

In [11]:
regression_results = results_df[results_df['task']=='regression']
classification_results = results_df[results_df['task']=='classification']

In [14]:
# regression ANova test

val_scores = []
unique_experiments = regression_results['exp'].unique()

for exp in unique_experiments:
    exp_column = regression_results[regression_results['exp']==exp]['val_score']
    val_scores.append(exp_column)
test = stats.f_oneway(*val_scores)
p_val = test.pvalue
print(p_val)

0.8011918700100025


For regression the p value of annova test is 0.8 > 0.05 menaing we cannot discard H0 hypothesis. There are no signs of the validation scores being sgnificantly different from each other on different configurations. 

In [16]:
# classif ANova test

val_scores = []
unique_experiments = classification_results['exp'].unique()

for exp in unique_experiments:
    exp_column = classification_results[classification_results['exp']==exp]['val_score']
    val_scores.append(exp_column)
test = stats.f_oneway(*val_scores)
p_val = test.pvalue
print(round(p_val,5))

0.0


For classification p value = 0, which is a sign that different configurations produce significanly different validation scores.